In [ ]:
# Uncomment kode di bawah ini jika terdapat masalah dengan Java di Windows

# import os
# import sys

# JAVA = r"C:\path\to\your\java\jdk" # Ganti dengan path JDK yang sesuai

# os.environ["JAVA_HOME"] = JAVA
# os.environ["PATH"] = os.path.join(JAVA, "bin") + os.path.pathsep + os.environ.get("PATH", "")
# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [ ]:
%pip install -r requirements.txt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim, lower

from pyspark.ml import Pipeline
from pyspark.ml.classification import LinearSVC
from pyspark.ml.feature import (
    IDF,
    HashingTF,
    Normalizer,
    StringIndexer,
    Tokenizer,
)

spark = SparkSession.builder \
    .appName("AmazonReviewsPolarity") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

df_train = spark.read.csv("../datasets/train.csv", header=True, inferSchema=True)
df_train = df_train.toDF("label", "title", "text")

df_test = spark.read.csv("../datasets/test.csv", header=True, inferSchema=True)
df_test = df_test.toDF("label", "title", "text")

In [ ]:
def preprocessing(df):
    df_clean = df.dropna()
    
    df_clean = df_clean.dropDuplicates(["title", "text"])
    
    df_clean = df_clean \
        .withColumn("title", trim(col("title"))) \
        .withColumn("text", trim(col("text")))
    
    invalid_words = ["n/a", "null", "undefined", "-"]
    
    df_clean = df_clean.filter(
        col("label").isin(1, 2) &
        (length(col("title")) >= 3) &
        (length(col("text")) >= 3) &
        (lower(col("text")).isin(invalid_words))
    )
    
    return df_clean

In [ ]:
tokenizer = Tokenizer(inputCol="text", outputCol="words")

hashingTF = HashingTF(
    inputCol="words", outputCol="rawFeatures", numFeatures=10000
)

idf = IDF(inputCol="rawFeatures", outputCol="idfFeatures")

normalizer = Normalizer(inputCol="idfFeatures", outputCol="features", p=2.0)

label_indexer = StringIndexer(inputCol="label", outputCol="label_index")

svm = LinearSVC(featuresCol="features", labelCol="label_index", maxIter=15)

pipeline = Pipeline(
    stages=[tokenizer, hashingTF, idf, normalizer, label_indexer, svm]
)

df_train = preprocessing(df_train)
model = pipeline.fit(df_train)

df_test = preprocessing(df_test)
predictions = model.transform(df_test)

predictions.select("text", "label_index", "prediction").show(100, truncate=30)